In [26]:
import numpy as np
import pandas as pd

files = {
    "customers": "Day 02/customers.csv",
    "products": "Day 02/products.csv",
    "orders": "Day 02/orders.csv",
    "order_items": "Day 02/order_items.csv",
    "payments": "Day 02/payments.csv",
    "shipments": "Day 02/shipments.csv",
    "returns": "Day 02/returns.csv"
}

data = {name: pd.read_csv(path)
        for name, path in files.items()}

for name, df in data.items():
    print(name, df.shape)
    print("======")
    print(df.isna().sum().sort_values(
        ascending=False).head())
    print("duplicates:", df.duplicated().sum())
    print(df.dtypes)
    print()

customers (5050, 13)
gender          301
loyalty_tier    266
age             150
phone           147
email           107
dtype: int64
duplicates: 50
customer_id         str
first_name          str
last_name           str
gender              str
age             float64
email               str
phone           float64
region_code         str
region_name         str
city                str
barangay            str
signup_date         str
loyalty_tier        str
dtype: object

products (800, 9)
stock_qty       19
product_id       0
product_name     0
category         0
brand            0
dtype: int64
duplicates: 0
product_id          str
product_name        str
brand               str
category            str
unit_cost       float64
unit_price          str
stock_qty       float64
uom                 str
launch_date         str
dtype: object

orders (15075, 8)
discount_pct    758
shipping_fee    474
order_id          0
customer_id       0
order_status      0
dtype: int64
duplicates: 75
order_i

In [54]:
data_clean = {
    f"{name}_clean": df.copy() 
    for name, df in data.items()
}
#data_clean['customers_clean']

In [60]:
# Show duplicate rows
# duplicate_rows = (
#     customers_clean[customers_clean.duplicated(keep=False)]
#     .sort_values(by=['customer_id'])
# )
# duplicate_rows

## 1
# Remove verified exact duplicates from customers, orders, and order_items.
data_clean["customers_clean"] = data_clean["customers_clean"].drop_duplicates()
data_clean["orders_clean"] = data_clean["orders_clean"].drop_duplicates()
data_clean["order_items_clean"] = data_clean["order_items_clean"].drop_duplicates()

## 2
# Clean currency fields in products, order_items, and payments.
def clean_currency_col(df, col_name):
    # Removes currency symbols, commas, and non-numeric characters, then converts to float.
    new_col_name = f"{col_name}_clean"
    if col_name in df.columns:
        df[new_col_name] = pd.to_numeric(
            df[col_name].astype(str).str.replace(r'[^0-9.\-]', '', regex=True),
            errors='coerce'
        )

# Define a dictionary mapping each table to its currency columns
datasets_to_clean = {
    "products_clean": ["unit_price", "unit_cost"],
    "order_items_clean": ["unit_price_at_purchase"],
    "payments_clean": ["amount_paid"]
}

# Loop through each dataset and its respective columns
for table_name, columns in datasets_to_clean.items():
    for col in columns:
        clean_currency_col(data_clean[table_name], col)

data_clean['payments_clean']



,payment_id,order_id,payment_method,payment_status,payment_date,amount_paid,amount_paid_clean
0,PAY0000001,ORD0000001,GCash,Paid,2024-05-09,2822.72,2822.72
1,PAY0000002,ORD0000002,COD,Paid,06/25/2024,2499.76,2499.76
2,PAY0000003,ORD0000003,GCash,Paid,2025-06-10,1983.66,1983.66
3,PAY0000004,ORD0000004,PayMaya,Failed,2025-06-21,1765.99,1765.99
4,PAY0000005,ORD0000005,COD,Pending,06/28/2024,1477.53,1477.53
...,...,...,...,...,...,...,...
14995,PAY0014996,ORD0014996,GCash,Paid,2025-06-07,1985.86,1985.86
14996,PAY0014997,ORD0014997,PayMaya,Paid,2025-12-03,1596.22,1596.22
14997,PAY0014998,ORD0014998,Bank Transfer,Paid,2026-01-05,836.13,836.13
14998,PAY0014999,ORD0014999,GCash,Paid,12/19/2025,NaN,NaN


In [31]:
data_clean['customers_clean']

,customer_id,first_name,last_name,gender,age,email,phone,region_code,region_name,city,barangay,signup_date,loyalty_tier
0,CUST000001,Grace,DelosReyes,NaN,20.0,grace.delosreyes95@outlook.com,9.196001e+09,Region III,Central Luzon,Angeles,Brgy 1,2025-12-31,Bronze
1,CUST000002,Ana,Ramos,Female,35.0,ana.ramos@companymail.ph,9.026542e+09,CARAGA,Caraga,Butuan,Brgy 9,2025-02-08,Bronze
2,CUST000003,Arvin,Navarro,Male,14.0,arvin.navarrooutlook.com,9.161850e+09,NCR,Metro Manila,Pasig,Brgy 2,2023-09-10,Bronze
3,CUST000004,Kyle,Rivera,F,26.0,kyle.rivera13@outlook.com,9.752553e+09,Region III,Central Luzon,Angeles,Brgy 1,2024-11-17,NaN
4,CUST000005,Juan,Mendoza,Male,29.0,juan.mendoza@gmail.com,9.350306e+09,Region III,Central Luzon,Angeles,Brgy 9,2023-01-21,Gold
...,...,...,...,...,...,...,...,...,...,...,...,...,...
5045,CUST000805,Maria,Fernandez,F,33.0,maria.fernandez49@outlook.com,9.755752e+09,Region VII,Central Visayas,Cebu City,Brgy 2,2023-02-27,Bronze
5046,CUST001442,Grace,Flores,F,41.0,graceflores@mailcom,9.253268e+09,Region III,Central Luzon,Angeles,Brgy 4,2025-11-27,Bronze
5047,CUST001095,Ana,Cruz,Unknown,25.0,ana.cruz@companymail.ph,9.218946e+09,Region IV-A,CALABARZON,San Pablo,Brgy 9,01/26/2025,Bronze
5048,CUST003989,Mia,Garcia,M,17.0,mia.garcia@mail.com,9.402610e+09,Region III,Central Luzon,San Fernando,Brgy 10,14-Oct-2024,Gold
